# Cilento Palaeocliffs — Machine Learning Analysis

**Academic group project.** Python machine-learning pipeline independently developed by Davide Passariello; GIS preparation and cartographic activities were collaborative.

Run with `dataset_training.csv` in the same directory. The optional `dataset_finale.csv` enables full-area prediction export for QGIS; it is **not** needed to train or evaluate the models.

The results use a stratified random hold-out split, **not** independent spatial validation.


**Import**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_curve, auc, confusion_matrix

**Caricamento dataset**

In [ ]:
from pathlib import Path

# Required for training and evaluation.
training_path = Path('dataset_training.csv')
df_train = pd.read_csv(training_path)

# Optional: required only for whole-area predictions exported to QGIS.
prediction_path = Path('dataset_finale.csv')
df_real = None
if prediction_path.exists():
    df_real = pd.read_csv(prediction_path, sep=';', decimal=',')
    df_real = df_real.rename(columns={'sar': 'surfaceare'})
    print(f'Geospatial dataset loaded: {len(df_real):,} rows')
else:
    print('dataset_finale.csv not found: training and evaluation will run normally; QGIS export will be skipped.')


**Pulizia e bilanciamento delle classi**

In [ ]:
df_train = df_train.dropna()
if df_real is not None:
    df_real = df_real.dropna()

plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df_train, x='Value', hue='Value', palette=['#2ecc71', '#e74c3c'], legend=False)
plt.title('Bilanciamento classi del training set', fontweight='bold')
plt.xticks(ticks=[0, 1], labels=['Falesia No (0)', 'Falesia Sì (1)'])
plt.ylabel('Numero di campioni')
for container in ax.containers:
    ax.bar_label(container)
plt.show()


**Rimozione valori che non sono caratteristiche**

In [ ]:
# Work on a separate dataframe so the original CSV-loaded data stays unchanged.
df_train_features = df_train.drop(columns=['FID', 'ID'])
df_train_features.head()


**Plot comparativo delle caratteristiche**

In [ ]:
sns.pairplot(df_train_features, hue='Value')
plt.show()


**Separazione feature/target e salvataggio coordinate per QGIS**

In [ ]:
# Non-destructive selection makes this cell safe to run repeatedly.
X = df_train_features.drop(columns=['Value'])
y = df_train_features['Value']

print('Dimensioni training:', X.shape, y.shape)

if df_real is not None:
    required = set(X.columns) | {'x', 'y', 'id'}
    missing = required - set(df_real.columns)
    if missing:
        raise ValueError(f'Missing columns in prediction dataset: {sorted(missing)}')
    X_coord = df_real['x']
    Y_coord = df_real['y']
    ID_coord = df_real['id']
    df_real_features = df_real[X.columns]


**Divisione del dataset**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    train_size=0.8,
                                                    shuffle=True,
                                                    stratify=y,
                                                    random_state=42)

print(f"Campioni di training: {X_train.shape[0]}")
print(f"Campioni di test: {X_test.shape[0]}")

**Normalizzazione di input**

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

if df_real is not None:
    df_real_features_scaled = scaler.transform(df_real_features)


**Random Forest**

In [ ]:
rf = RandomForestClassifier(class_weight='balanced', random_state=42)
rf.fit(X_train, y_train) #non scalati

**SVM**

In [ ]:
svm = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)
svm.fit(X_train_scaled, y_train)

**Valutazione, confronto e grafici**

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

y_pred_rf = rf.predict(X_test)
acc_rf, prec_rf, rec_rf = accuracy_score(y_test, y_pred_rf), precision_score(y_test, y_pred_rf), recall_score(y_test, y_pred_rf)

y_pred_svm = svm.predict(X_test_scaled)
acc_svm, prec_svm, rec_svm = accuracy_score(y_test, y_pred_svm), precision_score(y_test, y_pred_svm), recall_score(y_test, y_pred_svm)

#MATRICI DI CONFUSIONE
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
etichette_classi = ['Falesia No', 'Falesia Sì']

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, ax=axes[0], cmap='Blues', colorbar=False, display_labels=etichette_classi)
axes[0].set_title('Matrice di Confusione - Random Forest')

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_svm, ax=axes[1], cmap='Oranges', colorbar=False, display_labels=etichette_classi)
axes[1].set_title('Matrice di Confusione - SVM')

plt.tight_layout()
plt.show()

#RANDOM FOREST VS. SVM
labels = ['Accuracy', 'Precision', 'Recall']
x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
rects1 = ax.bar(x - width/2, [acc_rf, prec_rf, rec_rf], width, label='Random Forest', color='royalblue')
rects2 = ax.bar(x + width/2, [acc_svm, prec_svm, rec_svm], width, label='SVM', color='darkorange')

ax.set_ylabel('Punteggio')
ax.set_title('Confronto metriche: Random Forest vs SVM')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.1)
ax.legend(loc='lower right')
ax.bar_label(rects1, fmt='%.2f', padding=3)
ax.bar_label(rects2, fmt='%.2f', padding=3)

plt.tight_layout()
plt.show()

#CURVA ROC
y_prob_rf = rf.predict_proba(X_test)[:, 1]
y_prob_svm = svm.predict_proba(X_test_scaled)[:, 1]

fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
fpr_svm, tpr_svm, _ = roc_curve(y_test, y_prob_svm)

plt.figure(figsize=(8, 6))
plt.plot(fpr_rf, tpr_rf, color='royalblue', lw=2, label=f'Random Forest (AUC = {auc(fpr_rf, tpr_rf):.3f})')
plt.plot(fpr_svm, tpr_svm, color='darkorange', lw=2, label=f'SVM (AUC = {auc(fpr_svm, tpr_svm):.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')

plt.xlabel('Tasso di falsi positivi (FPR)')
plt.ylabel('Tasso di veri positivi (TPR)')
plt.title('Curva ROC - Confronto modelli')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

**Predizioni finali per QGIS**

In [ ]:
if df_real is None:
    print('Skipped QGIS prediction exports: dataset_finale.csv is not available.')
else:
    # Process the full study-area dataset and save georeferenced probabilities.
    predizioni_prob_rf = rf.predict_proba(df_real_features)[:, 1]
    predizioni_prob_svm = svm.predict_proba(df_real_features_scaled)[:, 1]

    df_rf = pd.DataFrame({
        'id': ID_coord, 'X': X_coord, 'Y': Y_coord,
        'Probabilita_RF': predizioni_prob_rf
    })
    file_output_rf = 'mappa_predizioni_RF.csv'
    df_rf.to_csv(file_output_rf, index=False)

    df_svm = pd.DataFrame({
        'id': ID_coord, 'X': X_coord, 'Y': Y_coord,
        'Probabilita_SVM': predizioni_prob_svm
    })
    file_output_svm = 'mappa_predizioni_SVM.csv'
    df_svm.to_csv(file_output_svm, index=False)

    print(f'1. {file_output_rf}')
    print(f'2. {file_output_svm}')
